# Ground Station Relevance
Feeds directly into Multi_Pass_Predictor requirements

- **Satellite set:** T9 (2023-174), 55 on-orbit candidates
- **Ground station:** Cal Poly SLO ($35.30^\circ$ N, $120.66^\circ$ W)
- **Minimum Elevation**: $\epsilon_{min}$
- **Propagation:** 24hr, 2000 timesteps from cache

## 1 - Passes per satellite
## 2 - Peak elevation distribution
## 3 - Pass duration distribution 
## 4 - Doppler shift range
## 5 - Pass window overlap 

## Key findings
## Implications for Multi_Pass_Predictor 



## 0 - Set Up

#### Imports, Constants, hist_df build

In [7]:
# Imports
import numpy as np
import json
import pandas as pd
from sgp4.api import Satrec
from astropy.time import Time
from astropy.coordinates import TEME, ITRS, EarthLocation, CartesianRepresentation
import astropy.units as u
import plotly.graph_objects as go
from pathlib import Path

# Constants 
R_EARTH = 6371.0  # Earth's radius in kilometers
MU_EARTH = 398600.4418  # Earth's gravitational parameter in km^3/s^2
INTLDES = '2023-174' 
LAUNCH_DATE = '2023-11-11'

# Ground Station 
GS_NAME    = 'Cal Poly SLO'
GS_LAT     = 35.30        # degrees North
GS_LON     = -120.66      # degrees East
GS_ALT     = 0.097        # km
MIN_ELEV   = 10.0         # degrees

# Propagation parameters
N_STEPS = 2000
N_HOURS = 24

# Load historical TLE data 
cache_path = Path.home() / 'mission_cache' / f'{INTLDES}_history.json'
with open(cache_path) as f:
    historical_tles = json.load(f)

print(f"Loaded {len(historical_tles)} historical TLE records")

# Build hist_df
launch_dt = pd.Timestamp(LAUNCH_DATE, tz='UTC')
records = []

for sat in tles_data:
    epoch = pd.Timestamp(sat['epoch'], tz='UTC')
    days = (epoch - launch_dt).total_seconds() / 86400
    records.append({
        'norad'             : int(sat['norad']),
        'name'              : sat['name'],  
        'epoch'             : epoch,
        'days_since_launch' : days,
        'line1'             : sat['line1'],
        'line2'             : sat['line2'],
    })
hist_df = pd.DataFrame(records).sort_values(
    ['norad', 'days_since_launch']
).reset_index(drop=True)

print(f"hist_df shape:            {hist_df.shape}")
print(f"Unique satellites:         {hist_df['norad'].nunique()}")
print(f"Days since launch range:   {hist_df['days_since_launch'].min():.1f}"
      f" - {hist_df['days_since_launch'].max():.1f}")


Loaded 5 historical TLE records
hist_df shape:            (9973, 6)
Unique satellites:         92
Days since launch range:   17.1 - 59.9


#### Day x TLE selection cell - first TLE per satellite at or after day 20

In [9]:
# Select first TLE per satellite at day x
TARGET_DAY = 20.0 

# Filter hist_df to get TLEs ASAP after TARGET_DAY
after_target = hist_df[hist_df['days_since_launch'] >= TARGET_DAY]

# Take first TLE per satelllite (earliest after day 20)
day_target_df = (after_target
                 .sort_values(['norad', 'days_since_launch'])
                 .groupby('norad')
                 .first()
                 .reset_index()) 

print(f"Target day:              {TARGET_DAY}")
print(f"Satellites with TLE at or after day {TARGET_DAY}: {len(day_target_df)}")
print(f"Days since launch range: {day_target_df['days_since_launch'].min():.1f}"
      f" - {day_target_df['days_since_launch'].max():.1f}")
print(f"\nSample:")
print(day_target_df[['norad','name','days_since_launch']].head(5))



Target day:              20.0
Satellites with TLE at or after day 20.0: 92
Days since launch range: 20.1 - 52.2

Sample:
   norad      name  days_since_launch
0  58256  OBJECT A          20.922526
1  58257  OBJECT B          20.860046
2  58258  OBJECT C          20.860819
3  58259  OBJECT D          20.859443
4  58260  OBJECT E          20.858568


#### Propagate Orbits (vectorize with SatArray)

In [24]:
from sgp4.api import SatrecArray 

# Build Satrec objects 
satrecs = []
valid_norads = []
valid_names = []

for _, row in day_target_df.iterrows():
    try:
        sat = Satrec.twoline2rv(row['line1'], row['line2'])
        satrecs.append(sat)
        valid_norads.append(row['norad'])
        valid_names.append(row['name'])
    except Exception as e:
        print(f"Failed NORAD {row['norad']}: {e}")

print(f"Succesfully built Satrec for {len(satrecs)} satellites")
print(f"First satellite epoch: "
      f"{satrecs[0].jdsatepoch + satrecs[0].jdsatepochF:.6f} JD")

# Build SatrecArray 
sats_array = SatrecArray(satrecs)

# Build time array from first satellite epoch
epoch0 = Time(satrecs[0].jdsatepoch + satrecs[0].jdsatepochF,
              format = 'jd', scale = 'utc')
dt_s   = (N_HOURS * 3600) / N_STEPS
times = [epoch0 + k * dt_s * u.s for k in range(N_STEPS)]

# Split JD for precision 
jd1_arr = np.array([t.jd1 for t in times])
jd2_arr = np.array([t.jd2 for t in times])

# Vectorized propagation
e_all, r_teme_all, v_teme_all = sats_array.sgp4(jd1_arr, jd2_arr)

print(f"Propagation complete")
print(f"r_teme_all shape:  {r_teme_all.shape}")
print(f"v_teme_all shape:  {v_teme_all.shape}")
print(f"Error codes != 0:  {(e_all != 0).sum()}")
print(f"NaN count:         {np.isnan(r_teme_all).sum()}")

Succesfully built Satrec for 92 satellites
First satellite epoch: 2460280.422526 JD
Propagation complete
r_teme_all shape:  (92, 2000, 3)
v_teme_all shape:  (92, 2000, 3)
Error codes != 0:  0
NaN count:         0
